# Lesson 05 — Reranking

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/05-reranking/colab.ipynb)

*Cheap retrieval builds a shortlist; an expensive model reorders it. A reranker reorders — it cannot conjure.*

**What you will learn**

- Bi-encoders vs cross-encoders: why one is fast and the other is accurate
- The over-fetch pattern and the **ceiling rule**: recall at shortlist depth bounds final quality
- Watching individual chunks move up and down under the reranker
- The decisive experiment: hybrid is a worse *ranker* but a better *candidate pool*
- A shortlist-depth sweep that overturned the conventional default — and a harness bug that once made it lie

| | |
|---|---|
| ⏱️ Time | 60 min |
| 🖥️ Runtime | CPU is fine (GPU makes the sweep faster) |
| 🤖 Needs the generator (Ollama) | no — retrieval only |

**Before this:** [04-hybrid-search-rrf](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/04-hybrid-search-rrf/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = False  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Bi-encoders and cross-encoders

```
BI-ENCODER (embedding model)                CROSS-ENCODER (reranker)

 query ──► [encoder] ──► q-vector             [CLS] query [SEP] passage [SEP]
                                ╲                         │
                                 cos(q, d) ──► score   [one transformer, attention
                                ╱                       across BOTH texts at once]
 doc ───► [encoder] ──► d-vector                          │
          (precomputed at ingest)                       score
```

| | Bi-encoder | Cross-encoder |
|---|---|---|
| Query and passage meet… | only at a dot product | inside every attention layer |
| Precompute documents? | yes — search is milliseconds | no — one model call per (query, passage) pair |
| Accuracy | good | much better |
| Run over the whole corpus? | yes | no — a million chunks per query is a stalled request |

Hence two stages: cheap retrieval produces a **shortlist**, the cross-encoder
**reorders** it.

In [ ]:
import time

import numpy as np
import pandas as pd

from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())
reranker = pipeline.reranker  # cross-encoder/ms-marco-MiniLM-L-6-v2, ~90MB
embedder = pipeline.embedder
print("reranker:", reranker.name)

### Score some pairs by hand

One question, one passage that *answers* it and one that is merely *on topic*. Compare
what each model thinks.

In [ ]:
from ragkit.types import Chunk

question = "Why can a reranker not fix poor retrieval recall?"
passages = {
    "answers it": "A reranker only reorders the shortlist it is given. If the relevant "
    "passage was not retrieved into the shortlist, no reranker can surface it.",
    "on topic": "Rerankers are cross-encoders that score a query and a passage together "
    "and are slower than embedding models.",
    "unrelated": "Chroma stores vectors on local disk and needs no separate server.",
}
q_vec = np.array(embedder.embed_query(question))
p_vecs = np.array(embedder.embed_documents(list(passages.values())))
chunks = [Chunk(chunk_id=k, doc_id=k, text=v) for k, v in passages.items()]
ce = dict((c.chunk_id, s) for c, s in reranker.rerank(question, chunks, top_k=3))

pd.DataFrame(
    {"bi-encoder cosine": p_vecs @ q_vec, "cross-encoder score": [ce[k] for k in passages]},
    index=list(passages),
).round(3)

Two things to notice:

- The cosine scores of "answers it" and "on topic" are close — both are *about*
  reranking. The cross-encoder reads the question and the passage together, so it can
  tell which one actually *answers*.
- Cross-encoder scores are **logits** — unbounded, can be negative. They rank well, but
  they are not probabilities and not comparable to cosine. (Lesson 12 finds they are
  worse than cosine for deciding "is there an answer at all?")

### What it costs

In [ ]:
if not pipeline._bm25.count():
    pipeline._bm25.load()
all_chunks = pipeline._bm25._chunks
timings = {}
for n in (5, 10, 25, 50, 100):
    started = time.perf_counter()
    reranker.rerank(question, all_chunks[:n], top_k=5)
    timings[n] = (time.perf_counter() - started) * 1000

started = time.perf_counter()
pipeline._store.search(embedder.embed_query(question), top_k=10)
dense_ms = (time.perf_counter() - started) * 1000

print(f"dense search over ALL {len(all_chunks)} chunks: {dense_ms:6.1f} ms")
for n, ms in timings.items():
    print(f"cross-encoder over {n:3d} chunks:       {ms:6.1f} ms")

Dense search over the whole corpus costs about the same as scoring a handful of chunks
with the cross-encoder, and the cross-encoder grows **linearly** with the number of
candidates. A hosted reranker bills per document scored. So the shortlist depth —
`rerank_candidates` — is both the quality ceiling and the cost dial.

## 2. The rule that matters most

> **Recall at the shortlist depth is a hard ceiling on final quality.**

A reranker reorders; it cannot conjure. If the answer is not in the top 10, no reranker
puts it in the top 5. When reranking disappoints, the problem is nearly always
shortlist depth or upstream recall — not the reranker.

Let's prove it on the golden set: for every answerable question, compare *did the
shortlist contain the answer?* with *is the answer in the final top 5?*

In [ ]:
from ragkit.eval.golden import load_golden
from ragkit.retrieve import Bm25Retriever, DenseRetriever, HybridRetriever
from ragkit.retrieve.rerank import RerankingRetriever

dense = DenseRetriever(pipeline._store, embedder)
lexical = Bm25Retriever(pipeline._bm25)
SHORTLIST = 5  # deliberately shallow, to make misses visible

hybrid = HybridRetriever([dense, lexical], candidates=SHORTLIST)
reranked = RerankingRetriever(hybrid, reranker, candidates=SHORTLIST)

rows = []
for q in [q for q in load_golden() if q.relevant]:
    in_shortlist = any(
        s.chunk.doc_id in q.relevant for s in hybrid.retrieve(q.question, top_k=SHORTLIST)
    )
    in_final = any(s.chunk.doc_id in q.relevant for s in reranked.retrieve(q.question, top_k=3))
    rows.append({"in shortlist": in_shortlist, "in final top 3": in_final})
table = pd.DataFrame(rows)
pd.crosstab(table["in shortlist"], table["in final top 3"])

The cell where **shortlist = False** and **final = True** is always **zero**. Not
usually — always, by construction. Every question the shortlist missed is lost no
matter how good the reranker is. That is the ceiling.

## 3. Watch chunks move

`RerankingRetriever` records `rank_before_rerank` on every result — the delta between
the two ranks is the only honest measure of what the reranker contributed.

In [ ]:
!{sys.executable} lessons/05-reranking/run.py "Why do I get charged twice?"

In [ ]:
results_frame(
    pipeline.retrieve("My spreadsheets turn into gibberish once they go in. Why?", top_k=5),
    chars=70,
)[["doc_id", "rank_before_rerank", "rerank", "dense_rank", "bm25_rank", "text"]]

## 4. Measure it — and run the decisive experiment

Lesson 04 ended with a hypothesis: hybrid produces a better *pool*, not a better
*ranking*. The test: **same reranker, different shortlist source.**

🔮 **Predict:** which will score better on `vocab_mismatch` recall@5 — dense → rerank,
or hybrid → rerank?

In [ ]:
from ragkit.eval.run import run_eval

runs = {}
for strategy in ["dense", "hybrid", "rerank"]:
    pipeline.set_strategy(strategy)
    runs[{"rerank": "hybrid → rerank"}.get(strategy, strategy)] = run_eval(
        pipeline=pipeline, retrieval_only=True, quiet=True
    )

candidates = pipeline.settings.rerank_candidates
pipeline._retriever = RerankingRetriever(dense, reranker, candidates=candidates)
runs["dense → rerank"] = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)
pipeline.set_strategy("rerank")

display(compare_runs(runs).round(3))
print("vocab_mismatch only:")
display(compare_runs(runs, kind="vocab_mismatch").round(3))

### What the repo measured

| Strategy | R@1 | R@3 | R@5 | nDCG@5 | vocab R@5 | ms |
|---|---|---|---|---|---|---|
| dense | 0.824 | 0.932 | 0.932 | 0.925 | 0.750 | 1078 |
| hybrid | 0.824 | 0.905 | 0.946 | 0.925 | 0.750 | 1048 |
| **hybrid → rerank** | 0.811 | **0.960** | **0.987** | **0.946** | **1.000** | 1613 |
| dense → rerank | — | 0.960 | 0.960 | 0.941 | 0.875 | — |

Confirmed: **hybrid search is worse than dense as a final ranker and better as a
candidate pool feeding a cross-encoder.** BM25 pulls documents into the shortlist that
dense alone misses; RRF orders them badly; the cross-encoder fixes the ordering.

That is the real argument for hybrid retrieval, and it is not the one usually given.
Neither technique is worth much alone on this corpus. Together they take vocab_mismatch
recall@5 from 0.750 to 1.000.

### R@1 dipped
Reranking is not free at the very top: the cross-encoder occasionally demotes a correct
rank-1 result. It buys a large gain at k=3 and k=5 for a small loss at k=1. If your
application shows exactly one passage, that trade may be bad. **Measure the k you
actually serve.**

## 5. The shortlist-depth sweep

`rerank_candidates` is the shortlist depth. The received wisdom is 25, and "deeper is at
least as good, just slower". 🔮 **Predict** the shape of the quality curve from depth 3
to 50.

In [ ]:
import matplotlib.pyplot as plt

deep_hybrid = HybridRetriever([dense, lexical], candidates=50)
sweep = {}
for depth in (3, 5, 10, 25, 50):
    pipeline._retriever = RerankingRetriever(deep_hybrid, reranker, candidates=depth)
    started = time.perf_counter()
    # deep_k=5 switches off the harness's depth-20 diagnostic — see section 6.
    summary = run_eval(pipeline=pipeline, retrieval_only=True, quiet=True, top_k=5, deep_k=5)
    summary["mean_latency_ms"] = (time.perf_counter() - started) * 1000 / len(summary["results"])
    sweep[depth] = summary
pipeline.set_strategy("rerank")

sweep_table = compare_runs({f"depth {d}": s for d, s in sweep.items()}).round(3)
display(sweep_table)

fig, ax1 = plt.subplots(figsize=(9, 3.5))
depths = list(sweep)
ax1.plot(depths, sweep_table["recall@5"], "o-", label="recall@5")
ax1.plot(depths, sweep_table["vocab R@5"], "s-", label="vocab R@5")
ax1.plot(depths, sweep_table["recall@1"], "^-", label="recall@1")
ax1.set_xscale("log")
ax1.set_xticks(depths, depths)
ax1.set_xlabel("shortlist depth")
ax2 = ax1.twinx()
ax2.bar(depths, sweep_table["ms/query"], alpha=0.15, width=[d * 0.3 for d in depths], color="grey")
ax2.set_ylabel("ms / query (bars)")
ax1.legend(loc="lower left")
plt.title("Quality vs cost by shortlist depth")
plt.show()

### What the repo's sweep showed

| depth | R@1 | R@3 | R@5 | nDCG@5 | vocab R@5 | ms/query |
|---|---|---|---|---|---|---|
| 3 | 0.838 | 0.960 | 0.973 | 0.946 | 0.875 | 258 |
| 5 | 0.838 | 0.960 | 0.973 | 0.946 | 0.875 | 263 |
| **10** | 0.811 | 0.960 | **0.987** | 0.946 | **1.000** | 466 |
| 25 | 0.811 | 0.960 | 0.987 | 0.944 | 1.000 | 1828 |
| 50 | 0.811 | 0.960 | **0.960** | 0.932 | 0.875 | 2831 |

Three things, none of them obvious:

1. **Depth 10 is the whole win** — it matches 25 at roughly a quarter of the latency. The
   repo's default (`rerank_candidates = 10`) was changed on this evidence.
2. **Depth 50 is *worse*, not just slower.** A small cross-encoder handed more
   distractors makes more mistakes, and some push a correct passage out of the top 5.
   "Deeper is monotonically safe" is false — the rule is that shortlist recall is a
   *ceiling*, not that deeper is better.
3. **Shallow depths have the best R@1** — fewer chances to demote the correct rank-1
   result. If you show exactly one passage, the right depth here is 3, not 10.

There is no single best depth — there is a best depth *for the k you actually serve*.
And with 44 questions, adjacent rows differ by one or two questions: **port the sweep,
not the number.**

## 6. The sweep that lied

The first run of this sweep showed quality *perfectly flat* from depth 5 to 100. That
was not a finding, it was a bug: `run_eval` always retrieved to depth 20 for its rank
diagnostic, and a reranker asked for 20 results must shortlist at least 20 — so every
configured depth below 20 was silently widened to 20 and measured the same pipeline.

Reproduce the bug by leaving the diagnostic on (`deep_k=None`):

In [ ]:
lying = {}
for depth in (3, 5, 10):
    pipeline._retriever = RerankingRetriever(deep_hybrid, reranker, candidates=depth)
    lying[f"depth {depth} (diagnostic on)"] = run_eval(
        pipeline=pipeline, retrieval_only=True, quiet=True, top_k=5
    )
pipeline.set_strategy("rerank")
compare_runs(lying).drop(columns=["ms/query"]).round(3)

Identical rows for different depths. The tell was that the result was **too clean**: a
genuinely flat curve across a 20× range should prompt suspicion, not a conclusion. The
measurement harness is also code, and it can lie with a perfectly plausible table.

## 7. Check your understanding

**1. Why can't you run the cross-encoder over the whole corpus instead of a shortlist?**

<details><summary>▶ Show answer</summary>

Nothing can be precomputed: it needs one forward pass per (query, passage) pair, so
cost grows linearly with corpus size per query. Dense search uses precomputed
vectors and an index, so it is effectively constant per query.

</details>

**2. Reranking disappoints on your system. What do you check first?**

<details><summary>▶ Show answer</summary>

Recall at the shortlist depth. If the right passages are not in the shortlist, the
reranker cannot help — deepen the shortlist or improve upstream retrieval (e.g.
hybrid). Use the harness's `first_relevant_rank` to tell "ranked 12th" from
"absent".

</details>

**3. Why did depth 50 score *worse* than depth 10?**

<details><summary>▶ Show answer</summary>

A small cross-encoder makes occasional mistakes; giving it more distractors gives
it more chances to rank a wrong passage above the right one, pushing the right one
out of the top 5.

</details>

## ✍️ Exercises

1. **Count the cost honestly.** From your sweep table, compute the extra milliseconds
   per query that reranking at depth 10 costs over plain hybrid. Inside a 3-second
   generation, does it matter? In an autocomplete box?
2. **Swap the model.** On a GPU runtime, try the stronger (2.3GB) BGE reranker and
   re-run section 4. Is it worth 25× the download? Measure, don't assume:

In [ ]:
# Exercise 2 — uncomment on a GPU runtime
# from ragkit.config import Settings
# from ragkit.providers import get_reranker
# strong = get_reranker(Settings(local_rerank_model="BAAI/bge-reranker-v2-m3"))
# pipeline._retriever = RerankingRetriever(HybridRetriever([dense, lexical], candidates=10),
#                                          strong, candidates=10)
# compare_runs({"MiniLM": runs["hybrid → rerank"],
#               "bge-reranker-v2-m3": run_eval(pipeline=pipeline, retrieval_only=True, quiet=True)})

## Key takeaways

1. Bi-encoders are fast and precomputable; cross-encoders read query and passage together and are far more accurate.
2. Recall at shortlist depth is a hard ceiling: a reranker reorders, it cannot conjure.
3. Hybrid → rerank beats dense → rerank: hybrid's value is the candidate pool.
4. Deeper shortlists are not monotonically better; measure the depth for the k you serve.
5. A too-clean result is a reason to check the harness — it once silently widened every shortlist.

## Next

**[Lesson 06 — Query transformation](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/06-query-transformation/colab.ipynb)**